In this file, the embedding from DLPFC data will be generated. SOMDE will be used to select the highly variable genes. If you don't have the data, Please run the somdeProcess.ipynb to generate the data. 

In [ ]:
# package test:

import torch
import random
import scanpy as sc
import pandas as pd
import anndata
import numpy as np
from CommonTrain_second_version import *

data4train = './data4train'

import scanpy as sc
import pandas as pd
import numpy as np
from somde import SomNode
import scipy.sparse
import os
import anndata
import matplotlib.pyplot as plt


# for loading DLPFC12 data
def load_DLPFC(root_dir='/home/salovjade/0324_raftupdata/DLPFC12', section_id='151507'):
    # 151507, ..., 151676 12 in total
    ad = sc.read_visium(path=os.path.join(root_dir, section_id), count_file=section_id+'_filtered_feature_bc_matrix.h5')
    ad.var_names_make_unique()

    gt_dir = os.path.join(root_dir, section_id, 'gt')
    gt_df = pd.read_csv(os.path.join(gt_dir, 'tissue_positions_list_GTs.txt'), sep=',', header=None, index_col=0)
    ad.obs['original_clusters'] = gt_df.loc[:, 6]
    keep_bcs = ad.obs.dropna().index
    ad = ad[keep_bcs].copy()
    ad.obs['original_clusters'] = ad.obs['original_clusters'].astype(int).astype(str)
    print(f"{section_id} loading done")

    return ad


# test by calling two functions
from _somdeprocess import load_DLPFC, compute_and_save_somde_csv
from _gene_cost_somde import compute_gene_cost_somde_pair
import scanpy as sc
import pandas as pd


section_ids_list = [
    ("151509", "151510"),
    ("151669", "151670"),
]

ROOT_DIR = "/home/salovjade/0324_raftupdata/DLPFC12"
SOMDE_DIR = "./data4train/somde"
OUT_DIR = "./outputEmbed_extra_9"
N_TOP_SV_GENES = 3000

for sid_A, sid_B in section_ids_list:

    print(f"\n=== Processing pair {sid_A} vs {sid_B} ===")

    # --------------------------------------------------
    # 1. load raw Visium slices
    # --------------------------------------------------
    sliceA = load_DLPFC(root_dir=ROOT_DIR, section_id=sid_A)
    sliceB = load_DLPFC(root_dir=ROOT_DIR, section_id=sid_B)

    # --------------------------------------------------
    # 2. preprocessing (normalize + log)
    # --------------------------------------------------
    for sl in (sliceA, sliceB):
        sc.pp.normalize_total(sl)
        sc.pp.log1p(sl)

    # --------------------------------------------------
    # 3. load SOMDE genes (already computed & saved)
    # --------------------------------------------------
    df_somde_A = pd.read_csv(f"{SOMDE_DIR}/somde_{sid_A}.csv")
    df_somde_B = pd.read_csv(f"{SOMDE_DIR}/somde_{sid_B}.csv")

    sv_genes_A = df_somde_A["g"].values[:N_TOP_SV_GENES]
    sv_genes_B = df_somde_B["g"].values[:N_TOP_SV_GENES]

    sliceA = sliceA[:, sv_genes_A].copy()
    sliceB = sliceB[:, sv_genes_B].copy()

    # --------------------------------------------------
    # 4. scale + PCA (node features)
    # --------------------------------------------------
    for sl in (sliceA, sliceB):
        sc.pp.scale(sl)
        sc.tl.pca(sl, n_comps=15)

    # --------------------------------------------------
    # 5. gene cost (ONE LINE 🔥)
    # --------------------------------------------------
    compute_gene_cost_somde_pair(
        sliceA=sliceA,
        sliceB=sliceB,
        section_id_A=sid_A,
        section_id_B=sid_B,
        n_h=100,
        n_epoch=3500,
        lr=0.0002,
        print_step=500,
        seed=0,
        device="cuda:0",
        output_dir=OUT_DIR,
    )

